# Full configuration interaction theory

Companion notebook to Chapter 5 of *Quantum mechanics for many-particle
systems*.  Every number quoted in the chapter is produced here.  The code is
the same as in `BookManybody/BookMaterial/Programs/fci.py`; the notebook runs
it section by section.

Contents:

1. Slater determinants as bit strings, and the excitation-level classification
2. The pairing model: full space versus the seniority-zero subspace
3. The block structure of the Hamiltonian matrix
4. Truncations: CIS, CID, CISD and full CI
5. Size consistency, and why coupled cluster repairs it
6. The Hubbard ring in the momentum basis
7. The exponential wall

## 1. Setup

We import the module directly, so that the notebook and the chapter can never
drift apart.  Adjust the path if you have moved the programs.

In [ ]:
import sys, os
import numpy as np

sys.path.insert(0, os.path.join("..", "BookManybody", "BookMaterial", "Programs"))
import fci

np.set_printoptions(precision=6, suppress=True, linewidth=120)

### Determinants as bit strings

A Slater determinant with $N$ particles in $n$ single-particle states is an
integer with $N$ bits set, exactly as in the last section of Chapter 3.
Creation and annihilation act by setting and clearing bits, and the sign is
$(-1)^{\ell}$ with $\ell$ the number of set bits below the one acted on.

In [ ]:
basis = fci.SlaterBasis(8, 4)          # 8 spin-orbitals, 4 particles
print("dimension:", basis.dim, "= C(8,4)")
for s in basis.states[:5]:
    print(format(s, "08b"), "  occupied:", [k for k in range(8) if s >> k & 1])

In [ ]:
state = 0b00101101
for p in (1, 4):
    sign, new = fci.create(state, p)
    if sign:
        print(f"a+_{p} |{state:08b}> = {sign:+d} |{new:08b}>")
    else:
        print(f"a+_{p} |{state:08b}> = 0   (orbital already occupied)")

### Excitation levels

Classifying every determinant by how many particles sit outside the reference
gives the distribution $1, 16, 36, 16, 1$ predicted by
$\binom{N}{k}\binom{n-N}{k}$ -- Exercise 1 of the chapter.

In [ ]:
reference = basis.states[0]
levels = basis.excitation_levels(reference)
for k in range(5):
    print(f"{k}p-{k}h: {(levels == k).sum():3d}")
print("total  :", basis.dim)

## 2. The pairing model

$$
\hat H = \xi\sum_{p\sigma}(p-1)\,a^{\dagger}_{p\sigma}a_{p\sigma}
      - \tfrac12 g\sum_{pq} a^{\dagger}_{p+}a^{\dagger}_{p-}a_{q-}a_{q+}
$$

Four doubly degenerate levels, four particles, $\xi = 1$.  The full space has
$\binom{8}{4} = 70$ determinants; the seniority-zero subspace used in
Chapter 4 has only $\binom{4}{2} = 6$.  Both give the same ground-state
energy.

In [ ]:
fci.demo_full_space()

In [ ]:
model = fci.PairingFCI(levels=4, n_particles=4, g=1.0, xi=1.0)
w, v = np.linalg.eigh(model.matrix())
print("ground-state energy, full 70-dimensional space :", f"{w[0]:.8f}")
print("Table 4.2, seniority-zero subspace, g = 1      :  0.63554847")

psi = v[:, 0]
odd = np.where(model.levels_of % 2 == 1)[0]
print("\nlargest amplitude on a broken-pair determinant :",
      f"{np.abs(psi[odd]).max():.2e}")

## 3. The block structure

A two-body operator connects determinants differing by at most two
single-particle states, so the matrix is banded in the excitation level.  For
the pairing model the seniority symmetry empties the odd blocks as well, and
the matrix is banded with a gap.

In [ ]:
fci.demo_block_structure()

## 4. Truncations

Keeping all determinants up to $n$ particle-hole pairs gives CIS, CID, CISD
and so on.  Each is a variational calculation in a subspace, so each energy is
an upper bound to the exact one, and the bounds fall monotonically as the
truncation is relaxed.

For the pairing model the singles do nothing at all -- the interaction cannot
break a pair -- and CISD coincides with CID.

In [ ]:
fci.demo_truncations()

## 5. Size consistency

Two identical non-interacting subsystems must give exactly twice the energy of
one.  Full CI does; CID does not, because two simultaneous double excitations
form a quadruple excitation of the combined reference.

This is the defect that coupled-cluster theory repairs, by writing
$e^{\hat T_2}$ instead of $1 + \hat C_2$: the term
$\tfrac12 \hat T_2^2$ then appears automatically, with its coefficient fixed
by the doubles rather than determined independently.

In [ ]:
fci.demo_size_consistency()

## 6. The Hubbard ring in the momentum basis

$$
\hat H = \sum_{k\sigma}\varepsilon_k\,c^{\dagger}_{k\sigma}c_{k\sigma}
 + \frac{U}{L}\sum_{kk'q} c^{\dagger}_{k+q\uparrow}c^{\dagger}_{k'-q\downarrow}
   c_{k'\downarrow}c_{k\uparrow},
\qquad \varepsilon_k = -2t\cos k .
$$

Six sites at half filling, dimension $\binom{6}{3}^2 = 400$.  Watch the
fraction of the correlation energy recovered by the doubles fall as $U/t$
grows: this is the onset of strong correlation, and it is where every method
built on a single reference determinant begins to fail.

In [ ]:
fci.demo_hubbard()

### Why CIS equals the reference, until it does not

Total crystal momentum is conserved, so a single excitation cannot be reached
from the Fermi sea: $\langle\Phi_0|\hat H|\Phi_i^a\rangle = 0$ for every $U$.
Brillouin's theorem here follows from a symmetry rather than from a
variational condition -- for a translationally invariant problem the
plane-wave determinant *is* the Hartree-Fock solution.

The CIS matrix is therefore block diagonal, and its lowest eigenvalue is the
smaller of the reference energy and the lowest eigenvalue of the singles
block.  At $U/t = 8$ the second one wins, and the "CIS ground state" is a
state orthogonal to the reference: a legitimate variational bound that no
longer describes the state we set out to compute.

In [ ]:
for U in (1.0, 2.0, 4.0, 8.0):
    h = fci.HubbardFCI(sites=6, n_up=3, n_down=3, t=1.0, U=U)
    M = h.matrix()
    r = h.index[h.reference]
    singles = np.where(h.levels_of == 1)[0]
    coupling = np.abs(M[r, singles]).max()
    lowest = np.linalg.eigvalsh(M[np.ix_(singles, singles)])[0]
    print(f"U/t = {U:4.1f}   max |<Phi_0|H|Phi_i^a>| = {coupling:8.2e}"
          f"   E_ref = {M[r, r]:9.6f}"
          f"   lowest singles eigenvalue = {lowest:9.6f}")

## 7. The exponential wall

$\dim\mathcal H = \binom{n}{N}$, and at half filling
$\binom{n}{n/2} \sim 2^n/\sqrt{n}$: the dimension doubles with every added
single-particle state.  Direct diagonalisation reaches about $10^5$, Lanczos
about $10^{10}$.  Beyond that the expansion itself must be truncated -- which
is what the remaining chapters are about.

In [ ]:
fci.demo_exponential_wall()

### The same wall, seen from four sides

| system | counting | dimension |
|---|---|---|
| Lipkin, $N$ particles, two levels | $\binom{2N}{N}$, or $\binom{N}{N/2}$ in the top multiplet | $\sim 2^N/\sqrt N$ |
| pairing, $n$ levels, $N$ particles | $\binom{2n}{N}$, or $\binom{n}{N/2}$ at seniority zero | exponential |
| Hubbard, $L$ sites | $\binom{L}{N_\uparrow}\binom{L}{N_\downarrow}$, at most $4^L$ | exponential |
| spin chain, $L$ sites | $\bigotimes_{i=1}^L\mathbb C^2$ | $2^L$ |

The tensor product is the fundamental statement; the binomial coefficient is
what remains after a conservation law has been imposed.  A symmetry removes a
factor, never the exponential.

What saves us is that physical ground states are not arbitrary vectors in
$\mathcal H$.  For gapped local Hamiltonians the entanglement across a cut
obeys an area law, the Schmidt spectrum decays quickly, and a matrix product
state with a modest bond dimension suffices.  The cell below measures the
Schmidt spectrum of the pairing ground state across a cut that separates the
two lowest levels from the two highest.

In [ ]:
model = fci.PairingFCI(levels=4, n_particles=4, g=1.0, xi=1.0)
w, v = np.linalg.eigh(model.matrix())
psi = v[:, 0]

occ = np.array([[(s >> k) & 1 for k in range(8)] for s in model.basis.states])
labels_A = [tuple(row[:4]) for row in occ]     # levels 1 and 2
labels_B = [tuple(row[4:]) for row in occ]     # levels 3 and 4
iA = {l: i for i, l in enumerate(sorted(set(labels_A)))}
iB = {l: i for i, l in enumerate(sorted(set(labels_B)))}

M = np.zeros((len(iA), len(iB)))
for c, (a, b) in enumerate(zip(labels_A, labels_B)):
    M[iA[a], iB[b]] = psi[c]

s = np.linalg.svd(M, compute_uv=False)
s = s[s > 1e-12]
print("Schmidt coefficients:", np.round(s, 6))
p = s**2
print("entanglement entropy:", f"{-np.sum(p * np.log2(p)):.6f}", "bits")
print("\nOnly a handful of Schmidt states carry any weight -- which is exactly")
print("what makes a compressed representation possible.")

## The full program

Everything above lives in `BookManybody/BookMaterial/Programs/fci.py`, which
runs as a script and prints all six demonstrations of the chapter.

In [ ]:
print(open(fci.__file__).read())